# Pillar 1 — Foundations explorer
Before/after view of every fix in Tasks 1.1 and 1.3. Run `etl_full.py` first.

In [1]:
import os, sys
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "datasets")):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
R = "outputs/results/mohammed_faisal"
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 30)
print("repo root:", ROOT)

repo root: C:\Users\mohammed.faisal\OneDrive - Presight AI\Desktop\stackup-engineering-academy_assessment


## Task 1.1 — projects_clean

In [2]:
p = pd.read_csv(f"{R}/01_foundations/projects_clean.csv")
print(p.shape); p.head()

(500, 17)


,project_id,project_name,department,status,start_date,end_date,budget,actual_cost,project_manager_id,priority,region,budget_variance,is_over_budget,duration_days,budget_utilisation_pct,status_category,risk_level
0,PRJ0001,MLOps Platform,Sustainability,Completed,2023-06-13,2023-12-01,1200000.0,907809.0,EMP0126,Medium,Dubai,-292191.0,False,171.0,75.65,Closed,Low
1,PRJ0002,Tax Automation Platform - Q2,Marketing,In Progress,2025-02-28,NaN,800000.0,795986.0,EMP0045,Medium,Abu Dhabi,-4014.0,False,NaN,99.50,Active,Medium
2,PRJ0003,ERP Consolidation Phase 2,Legal,Completed,2023-11-21,2024-12-16,80000.0,56981.0,EMP0405,Medium,Dubai,-23019.0,False,391.0,71.23,Closed,Low
3,PRJ0004,Employee Wellness App - Q1,Marketing,Completed,2024-06-09,2025-08-17,800000.0,707314.0,EMP0270,Medium,Abu Dhabi,-92686.0,False,434.0,88.41,Closed,Low
4,PRJ0005,Mobile Workforce App - Q4,Finance,In Progress,2023-04-25,NaN,2000000.0,1329426.0,EMP0478,High,Ras Al Khaimah,-670574.0,False,NaN,66.47,Active,Medium


In [3]:
p.groupby(["status_category","risk_level"]).size().unstack(fill_value=0)

risk_level,High,Low,Medium
status_category,,,
Active,93,40,56
Closed,86,70,58
Pending,9,57,31


In [4]:
p[["budget_variance","budget_utilisation_pct","duration_days"]].describe().round(2)

,budget_variance,budget_utilisation_pct,duration_days
count,440.00,440.00,214.00
mean,-66326.35,85.53,292.87
std,166901.53,24.81,141.96
min,-723621.00,10.44,60.00
25%,-109485.25,71.37,164.50
50%,-19240.50,88.38,272.50
75%,9475.75,105.07,423.25
max,381747.00,119.68,534.00


## Task 1.3 — employees: raw vs clean

In [5]:
raw = pd.read_csv("datasets/employees.csv", dtype={"hire_date": "string"})
clean = pd.read_csv(f"{R}/01_foundations/employees_clean.csv")
print(raw.shape, clean.shape)

(1000, 12) (1000, 13)


In [6]:
# Fix 1: missing emails -> derived
m = raw["email"].isna()
pd.DataFrame({"raw_email": raw.loc[m,"email"], "clean_email": clean.loc[m,"email"], "name": raw.loc[m,"full_name"]})

,raw_email,clean_email,name
357,NaN,anika.malik@presight.ai,Anika Malik
372,NaN,hamad.saleh@presight.ai,Hamad Saleh
460,NaN,mohammed.al.marri@presight.ai,Mohammed Al Marri
535,NaN,nina.laurent@presight.ai,Nina Laurent
646,NaN,salma.al.shamsi.emp0647@presight.ai,Salma Al Shamsi
657,NaN,saif.al.mazrouei.emp0658@presight.ai,Saif Al Mazrouei
703,NaN,omar.al.mansoori.emp0704@presight.ai,Omar Al Mansoori
790,NaN,anna.hernandez@presight.ai,Anna Hernandez
849,NaN,divya.mehta@presight.ai,Divya Mehta
851,NaN,mateo.reddy.emp0852@presight.ai,Mateo Reddy


In [7]:
# Fix 2: unparseable hire dates -> NULL + review flag
bad = pd.to_datetime(raw["hire_date"], errors="coerce").isna()
pd.DataFrame({"raw": raw.loc[bad,"hire_date"], "clean": clean.loc[bad,"hire_date"], "flags": clean.loc[bad,"dq_flags"]})

,raw,clean,flags
37,-999,NaN,hire_date_invalid;active_without_hire_date_REVIEW
159,99999-01-01,NaN,hire_date_invalid;active_without_hire_date_REVIEW
174,-999,NaN,hire_date_invalid;active_without_hire_date_REVIEW
394,99999-01-01,NaN,hire_date_invalid;active_without_hire_date_REVIEW
550,99999-01-01,NaN,hire_date_invalid;active_without_hire_date_REVIEW
599,-999,NaN,hire_date_invalid;active_without_hire_date_REVIEW
854,-999,NaN,hire_date_invalid;active_without_hire_date_REVIEW
972,-999,NaN,hire_date_invalid;active_without_hire_date_REVIEW


In [8]:
# Fix 3: negative experience and out-of-band salary
d = raw[(raw.years_experience<0)|(raw.salary>raw.groupby("level").salary.transform("quantile",0.99)*1.5)][["employee_id","level","salary","years_experience"]]
d.merge(clean[["employee_id","salary","years_experience","dq_flags"]], on="employee_id", suffixes=("_raw","_clean"))

,employee_id,level,salary_raw,years_experience_raw,salary_clean,years_experience_clean,dq_flags
0,EMP0216,Junior,12895,-1,12895,1,experience_imputed
1,EMP0356,Junior,76507,-1,15301,1,experience_imputed;salary_corrected
2,EMP0445,Senior,33530,-1,33530,9,experience_imputed
3,EMP0583,Senior,35791,-1,35791,9,experience_imputed
4,EMP0720,Mid,18654,-1,18654,4,experience_imputed


In [9]:
# Rows touched per fix type
clean["dq_flags"].dropna().str.split(";").explode().value_counts()

dq_flags
email_derived                      10
hire_date_invalid                   8
active_without_hire_date_REVIEW     8
manager_placeholder_nulled          5
experience_imputed                  5
salary_corrected                    3
Name: count, dtype: int64

In [10]:
print(open(f"{R}/01_foundations/pipeline_summary.txt").read()[:2500])

PRESIGHT ETL PIPELINE SUMMARY
Run timestamp     : 2026-09-21T13:36:57
Pipeline duration : 0.91 s

ROW COUNTS (raw -> clean)
------------------------------------------------------------
  projects       raw=    500   clean=    500
  employees      raw=   1000   clean=   1000
  transactions   raw=  50000   clean=  50000

DATA QUALITY FIXES (rows affected)
------------------------------------------------------------
  projects: null budget -> 0                                           32
  projects: null actual_cost -> 0                                      60
  employees: missing employee_id (unfixable)                            0
  employees: missing full_name (unfixable)                              0
  employees: missing department (unfixable)                             0
  employees: missing role (unfixable)                                   0
  employees: missing level (unfixable)                                  0
  employees: missing status (unfixable)                          